In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [2]:
train = pd.read_csv(r"E:\ICBT\CI Kaggle competition\predict-housing-price-2\estate_train.csv")
test = pd.read_csv(r"E:\ICBT\CI Kaggle competition\predict-housing-price-2\estate_test.csv")
sample_sub = pd.read_csv(r"E:\ICBT\CI Kaggle competition\predict-housing-price-2\estate_sample_submission.csv")

train.head()

,IncomeLevel,PropertyAge,TotalRooms,TotalBedrooms,NeighborhoodPop,AvgOccupancy,Latitude,Longitude,TargetPrice,PropertyID,RoomsPerHousehold,BedroomsRatio
0,3.287977,32.970198,5.128150,0.990769,2339.474039,3.739113,32.71,-117.03,1.030,PROP_14196,1.359130,0.200576
1,3.804601,49.030192,4.372696,1.040469,1269.383596,1.429576,33.77,-118.16,3.821,PROP_08267,2.573820,0.232703
2,4.193302,NaN,5.718626,0.989809,865.436104,2.481219,34.66,-120.48,1.726,PROP_17445,2.073224,0.174486
3,2.029509,36.255617,4.131418,1.032285,1455.381923,3.914447,32.69,-117.11,0.934,PROP_14265,1.002116,0.258269
4,3.540823,42.918534,6.270531,1.147146,895.050628,2.681969,36.78,-119.80,0.965,PROP_02271,2.725400,0.180940


In [3]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nColumns:", train.columns.tolist())
print("\nData types:\n", train.dtypes)
print("\nFirst rows:\n", train.head())
print("\nMissing values:\n", train.isnull().sum().sort_values(ascending=False).head(20))
print("\nTarget stats:\n", train['TargetPrice'].describe())
print("\nSample submission format:\n", sample_sub.head())

Train shape: (16512, 12)
Test shape: (4128, 11)

Columns: ['IncomeLevel', 'PropertyAge', 'TotalRooms', 'TotalBedrooms', 'NeighborhoodPop', 'AvgOccupancy', 'Latitude', 'Longitude', 'TargetPrice', 'PropertyID', 'RoomsPerHousehold', 'BedroomsRatio']

Data types:
 IncomeLevel          float64
PropertyAge          float64
TotalRooms           float64
TotalBedrooms        float64
NeighborhoodPop      float64
AvgOccupancy         float64
Latitude             float64
Longitude            float64
TargetPrice          float64
PropertyID               str
RoomsPerHousehold    float64
BedroomsRatio        float64
dtype: object

First rows:
    IncomeLevel  PropertyAge  TotalRooms  TotalBedrooms  NeighborhoodPop  \
0     3.287977    32.970198    5.128150       0.990769      2339.474039   
1     3.804601    49.030192    4.372696       1.040469      1269.383596   
2     4.193302          NaN    5.718626       0.989809       865.436104   
3     2.029509    36.255617    4.131418       1.032285      145

In [4]:
import numpy as np
from sklearn.cluster import KMeans

train["PropertyAge"] = train["PropertyAge"].fillna(train["PropertyAge"].median())
test["PropertyAge"] = test["PropertyAge"].fillna(train["PropertyAge"].median())

def add_features(df):
    df = df.copy()
    df["RoomsPerPop"] = df["TotalRooms"] / df["NeighborhoodPop"]
    df["BedroomsPerRoom"] = df["TotalBedrooms"] / df["TotalRooms"]
    df["PopPerHousehold"] = df["NeighborhoodPop"] / (df["TotalRooms"] / df["RoomsPerHousehold"])
    df["IncomePerRoom"] = df["IncomeLevel"] / df["RoomsPerHousehold"]
    df["IncomeSq"] = df["IncomeLevel"] ** 2
    df["IncomeXRooms"] = df["IncomeLevel"] * df["RoomsPerHousehold"]
    df["LatLong"] = df["Latitude"] * df["Longitude"]
    df["DistToCoastProxy"] = np.sqrt((df["Latitude"] - 35.0) ** 2 + (df["Longitude"] + 119.0) ** 2)
    return df

train = add_features(train)
test = add_features(test)

# KMeans on lat/long fitted on train only, then used to add geo-cluster distance
# features to both train and test -- captures location effects on price without leakage.
N_CLUSTERS = 75
geo_kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(train[["Latitude", "Longitude"]])

def add_geo_clusters(df):
    df = df.copy()
    dists = geo_kmeans.transform(df[["Latitude", "Longitude"]])
    df["GeoCluster"] = dists.argmin(axis=1)
    dist_cols = pd.DataFrame(dists, columns=[f"DistCluster{i}" for i in range(N_CLUSTERS)], index=df.index)
    return pd.concat([df, dist_cols], axis=1)

train = add_geo_clusters(train)
test = add_geo_clusters(test)

features = ["IncomeLevel", "PropertyAge", "TotalRooms", "TotalBedrooms",
            "NeighborhoodPop", "AvgOccupancy", "Latitude", "Longitude",
            "RoomsPerHousehold", "BedroomsRatio",
            "RoomsPerPop", "BedroomsPerRoom", "PopPerHousehold", "IncomePerRoom",
            "IncomeSq", "IncomeXRooms", "LatLong", "DistToCoastProxy", "GeoCluster"] + \
           [f"DistCluster{i}" for i in range(N_CLUSTERS)]

x_train = train[features]
x_test = test[features]
y_train = train["TargetPrice"]


In [5]:
model = LinearRegression()
model.fit(x_train, y_train)
predictions = model.predict(x_test)


In [6]:
from sklearn.model_selection import train_test_split
x_tr, x_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)


In [7]:
model.fit(x_tr, y_tr)
val_preds = model.predict(x_val)
from sklearn.metrics import mean_squared_error
import numpy as np
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
print(rmse)


0.8981263215322364


In [8]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(x_tr, y_tr)
rf_val_preds = rf_model.predict(x_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_val_preds))
print("RandomForest RMSE:", rf_rmse)


RandomForest RMSE: 0.46129244976230166


In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(max_iter=600, learning_rate=0.04, max_leaf_nodes=63, random_state=42)
hgb_model.fit(x_tr, y_tr)
hgb_val_preds = hgb_model.predict(x_val)
hgb_rmse = np.sqrt(mean_squared_error(y_val, hgb_val_preds))
print("HistGradientBoosting RMSE:", hgb_rmse)

print("\nComparison:")
print("Linear Regression RMSE:  ", rmse)
print("Random Forest RMSE:      ", rf_rmse)
print("HistGradientBoosting RMSE:", hgb_rmse)


HistGradientBoosting RMSE: 0.4545257854087379

Comparison:
Linear Regression RMSE:   0.8981263215322364
Random Forest RMSE:       0.46129244976230166
HistGradientBoosting RMSE: 0.4545257854087379


In [10]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=1200, learning_rate=0.02, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_lambda=1.0, random_state=42, n_jobs=-1
)
xgb_model.fit(x_tr, y_tr)
xgb_val_preds = xgb_model.predict(x_val)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_val_preds))
print("XGBoost RMSE:", xgb_rmse)

print("\nComparison:")
print("Linear Regression RMSE:  ", rmse)
print("Random Forest RMSE:      ", rf_rmse)
print("HistGradientBoosting RMSE:", hgb_rmse)
print("XGBoost RMSE:             ", xgb_rmse)


XGBoost RMSE: 0.43997750515523426

Comparison:
Linear Regression RMSE:   0.8981263215322364
Random Forest RMSE:       0.46129244976230166
HistGradientBoosting RMSE: 0.4545257854087379
XGBoost RMSE:              0.43997750515523426


## Neural Network (MLPRegressor)

Adding a neural network model for comparison. MLPs are sensitive to feature scale, so inputs are standardized before training.

In [11]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_tr_scaled = scaler.fit_transform(x_tr)
x_val_scaled = scaler.transform(x_val)

mlp_model = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu",
                          max_iter=1000, random_state=42, early_stopping=True)
mlp_model.fit(x_tr_scaled, y_tr)
mlp_val_preds = mlp_model.predict(x_val_scaled)
mlp_rmse = np.sqrt(mean_squared_error(y_val, mlp_val_preds))
print("MLPRegressor RMSE:", mlp_rmse)

print("\nComparison:")
print("Linear Regression RMSE:  ", rmse)
print("Random Forest RMSE:      ", rf_rmse)
print("HistGradientBoosting RMSE:", hgb_rmse)
print("MLPRegressor RMSE:        ", mlp_rmse)

MLPRegressor RMSE: 0.5281842463740993

Comparison:
Linear Regression RMSE:   0.8981263215322364
Random Forest RMSE:       0.46129244976230166
HistGradientBoosting RMSE: 0.4545257854087379
MLPRegressor RMSE:         0.5281842463740993


In [12]:
# Refit the best-performing model (tuned XGBoost, based on validation RMSE above) on the full training data.
best_model = XGBRegressor(
    n_estimators=1200, learning_rate=0.02, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_lambda=1.0, random_state=42, n_jobs=-1
)
best_model.fit(x_train, y_train)
test_preds = best_model.predict(x_test)

submission = pd.DataFrame({
    "PropertyID": test["PropertyID"],
    "TargetPrice": test_preds
})
submission.to_csv("submission-Nilucshiha.csv", index=False)
